## Leverage Document Grounding in Orchestration Service for RAG-based Content Generation

In this learning journey, you will learn how to leverage the Document Grounding module in the Orchestration Service to generate content using the Retrieval-Augmented Generation (RAG) approach.
The Document Grounding module helps in grounding the input questions to relevant documents.
The grounding process involves retrieving relevant documents from a knowledge base and using them to high-quality generate responses.
The knowledge base can be a collection of documents in a sharepoint folder, aws s3, an elastic search engine, or data repository which contains vectors.

In this learning journey, you will perform the following steps:
- Create the knowledge base with the relevant documents.
- Configure the Document Grounding module in the Orchestration Service.
- Generate content based on the knowledge base using the RAG approach.


## Prerequisites
Install the Generative AI Hub SDK using the following command:

In [ ]:
%pip install "sap-ai-sdk-gen[all]"

### Authenticating AI Core

In [1]:
import json
import os
from ai_core_sdk.ai_core_v2_client import AICoreV2Client

with open("creds.json") as f:
    credCF = json.load(f)

# Set AI Core env vars (NO resource group yet)
os.environ["AICORE_AUTH_URL"] = credCF["url"] + "/oauth/token"
os.environ["AICORE_CLIENT_ID"] = credCF["clientid"]
os.environ["AICORE_CLIENT_SECRET"] = credCF["clientsecret"]
os.environ["AICORE_BASE_URL"] = credCF["serviceurls"]["AI_API_URL"] + "/v2"

ai_core_client = AICoreV2Client(
    base_url=os.environ["AICORE_BASE_URL"],
    auth_url=os.environ["AICORE_AUTH_URL"],
    client_id=os.environ["AICORE_CLIENT_ID"],
    client_secret=os.environ["AICORE_CLIENT_SECRET"]
)


### Resource Group

This step creates a new resource group in SAP AI Core and tags it with a label (document-grounding) to logically group related resources. The access token is used for authorized API access.

In [2]:
from ai_core_sdk.models.resource_group import Label

# Name of the resource group to create
resource_group = "rg-test1" 

labels = [
    Label(
        key="ext.ai.sap.com/document-grounding",
        value="true"
    )
]

# Create Resource Group
try:
    rg = ai_core_client.resource_groups.create(
        resource_group_id = resource_group,
        labels = labels
    )
    print("Created resource group:", rg.resource_group_id)
except Exception as e:
    if "already exists" in str(e):
        print(f"Resource group '{resource_group}' already exists")
    else:
        raise


Created resource group: rg-test1


### AI Core client key

In [3]:
# Set Resource Group context
os.environ["AICORE_RESOURCE_GROUP"] = resource_group

scoped_ai_core_client = AICoreV2Client(
    base_url=os.environ["AICORE_BASE_URL"],
    auth_url=os.environ["AICORE_AUTH_URL"],
    client_id=os.environ["AICORE_CLIENT_ID"],
    client_secret=os.environ["AICORE_CLIENT_SECRET"],
    resource_group=os.environ["AICORE_RESOURCE_GROUP"]
)


### configuration and Deployment

This step creates a configuration for an LLM orchestration scenario in SAP AI Core using the given executableId and scenarioId. The loop ensures the config is retried until it successfully returns a 201 Created status, handling transient errors.

In [4]:
# Define scenario ID, executable ID, and configuration suffix 
scenario_id = "orchestration" 
executable_id = "orchestration" 
config_suffix = "config-new" # Enter your configuration name 
config_name = f"{config_suffix}-orchestration" 

# Create a new configuration 
config = scoped_ai_core_client.configuration.create( 
    scenario_id=scenario_id, 
    executable_id=executable_id, 
    name=config_name 
) 
print(f"Configuration created successfully with ID: {config.id} and Name: {config_name}") 

Configuration created successfully with ID: 4db27fe7-0cb4-4fd6-a189-25d519177350 and Name: config-new-orchestration


This step deploys the LLM configuration. It then waits until the deployment is ready and retrieves the deploymentUrl(orchestration url), which is used to trigger orchestration requests.

In [5]:
# Create a deployment using the configuration ID from the previous cell 

deployment = scoped_ai_core_client.deployment.create(configuration_id=config.id) 
print(f"Deployment created successfully with ID: {deployment.id}") 

Deployment created successfully with ID: d046384b136e97c9


### Poll until the deployment is ready

In [7]:
import time

while True:
    deployment_details = scoped_ai_core_client.deployment.get(
        deployment_id=deployment.id
    )

    status = deployment_details.status
    print("Deployment status:", status)

    if status == "RUNNING":
        break

    time.sleep(10)


Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.UNKNOWN
Deployment status: Status.PENDING
Deployment status: Status.PENDING
Deployment status: Status.PENDING
Deployment status: Status.PENDING
Deployment status: Status.PENDING
Deployment status: Status.PENDING
Deployment status: Status.RUNNING
Deployment status: Status.RUNNING
Deployment status: Status.RUNNING
Deployment status: Status.RUNNING
Deployment status: Status.RUNNING


KeyboardInterrupt: 

Here, you are explicitly defining the orchestration service deployment URL (orchestration_service_url) which points to your deployed LLM configuration. This URL is used to send inference requests (like prompt executions) to the SAP AI Core Orchestration.

### Generic Secret

#### In this tutorial, we're demonstrating how to create a vector knowledge base by connecting either SharePoint or AWS S3 as the document source—multiple options are supported and optional based on your setup.

#### creating knowledge base using Sharepoint - option 1

This step specifically creates a secret in SAP AI Core that stores Base64-encoded credentials for SharePoint access, securely enabling document grounding workflows via Microsoft Graph.

In [ ]:
json_data = {
    'name': '<generic secret name>',
    'data': {
        'description': '<description of generic secret>',
        'clientId': '<client id>',
        'authentication': 'T0F1dGgyUGFzc3dvcmQ=',
        'tokenServiceURL': '<token service url>',
        'password': '<password>',
        'url': 'aHR0cHM6Ly9ncmFwaC5taWNyb3NvZnQuY29t',
        'tokenServiceURLType': 'RGVkaWNhdGVk',
        'user': '<user>',
        'clientSecret': '<client secret>',
        'scope': 'aHR0cHM6Ly9ncmFwaC5taWNyb3NvZnQuY29tLy5kZWZhdWx0',
    },
    'labels': [
        {
            'key': 'ext.ai.sap.com/document-grounding',
            'value': 'true',
        },
    ],
}

secret = requests.post(f'{AI_API_URL}/v2/admin/secrets', headers=headers, json=json_data)

secret.json()

#### creating knowledge base using AWS S3 - Option 2

Alternatively, instead of SharePoint, we can use AWS S3 as a document repository for grounding. In the example below, we securely store credentials as a secret named aws-s3-secret that will later be referenced in the pipeline creation.

This makes it clear that both SharePoint and AWS S3 are optional approaches and interchangeable based on the user’s infrastructure.

In [ ]:

# Prepare secret payload
secret_payload = {
    "name": "<generic secret name>",
    "data": {  
        "description": "<description of generic secret>",
        "url": "<url>",
        "authentication": "Tm9BdXRoZW50aWNhdGlvbg==",
        "access_key_id": "<access key id>",
        "secret_access_key": "<secret access key>",
        "bucket": "<bucket>",
        "region": "<region>",
        "host": "<host>",
        "username": "<username>"
    },
    "labels": [
        {
            "key": "ext.ai.sap.com/document-grounding",
            "value": "true"
        },
         {
            "key": "ext.ai.sap.com/documentRepositoryType",
            "value": "S3"
        }
    ]
}

# Create secret
response = requests.post(f"{AI_API_URL}/v2/admin/secrets", headers=headers, json=secret_payload)
print("Secret creation:", response.status_code, response.text)


### Pipeline Creation

#### Pipeline creation using sharepoint - option 1
In this step, we are creating a document grounding pipeline using SharePoint as the knowledge source. The pipeline connects to the document repository defined in the SharePoint site using the previously created secret 

In [ ]:
json_data = {
    'type': 'MSSharePoint',
    'configuration': {
        'destination': '<generic secret name>',
        'sharePoint': {
            'site': {
                'name': 'Dev_blr3_document',
                "includePaths": [
          "/sample_emails/output_texts"
        ]
            },
        },
    },
}

while True:
    pipeline = requests.post(f'{AI_API_URL}/v2/lm/document-grounding/pipelines', headers=headers, json=json_data)
    if(pipeline.status_code == 201):
        break

pipeline.json()['pipelineId']

#### Pipeline creation using AWS S3 - option 2
Once the secret (aws-s3-secret) is created, we can now configure the document grounding pipeline using AWS S3 as the data source. This example shows how to set up a pipeline by referencing the created secret. The pipeline will extract and prepare documents from the specified S3 bucket for grounding.

🔄 You can follow a similar flow for SharePoint or other supported sources — choosing between SharePoint and S3 is flexible based on your document storage setup.

In [6]:
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.document_grounding.client import PipelineAPIClient
from gen_ai_hub.document_grounding.models.pipeline import S3PipelineCreateRequest, CommonConfiguration

aicore_client = get_proxy_client()
pipelines_api_client = PipelineAPIClient(aicore_client)
generic_secret_s3_bucket = "aws-s3-secretnew"
s3_config = S3PipelineCreateRequest(configuration=CommonConfiguration(destination=generic_secret_s3_bucket))
response = pipelines_api_client.create_pipeline(s3_config)
print(f"Reference the Vector knowledge base using the pipeline ID: {response.pipelineId}")
# check the status of the vectorization pipeline until it is completed
print(pipelines_api_client.get_pipeline_status(response.pipelineId))

Reference the Vector knowledge base using the pipeline ID: 6e81abec-40cb-4c54-8c40-d9e0bdcf491d
lastStarted='' status='NEW'


#### Set Up the Orchestration Service

Now that we have our document grounding pipeline ready, we can configure the LLM Orchestration Service to process incoming user queries in context.

We define a system message to describe the business scenario for the LLM — in this case, a Facility Solutions Company offering property maintenance and support services. The prompt template includes placeholders for the user’s query and the grounded document context (retrieved from S3 or SharePoint), making the responses personalized and context-aware.

💡 This setup ensures that the LLM generates accurate, domain-specific, and grounded responses using the extracted content from your enterprise documents.

In [29]:

from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.orchestration_v2.models.message import SystemMessage, UserMessage
from gen_ai_hub.orchestration_v2.models.template import Template
from gen_ai_hub.orchestration_v2.service import OrchestrationService

# Set up Orchestration Service (V2)
proxy_client = get_proxy_client()
orchestration_service = OrchestrationService(proxy_client)

# Runtime input for the orchestration pipeline
template = Template(
    template=[
            SystemMessage(content="""Facility Solutions Company provides services to luxury residential complexes, 
                apartments, individual homes, and commercial properties such as office buildings, 
                retail spaces, industrial facilities, and educational institutions. 
                Customers are encouraged to reach out with maintenance requests, service deficiencies, 
                follow-ups, or any issues they need by email."""),
        UserMessage(content="""You are a helpful assistant for any queries for answering questions. 
                Answer the request by providing relevant answers that fit to the request.\n\n
                Request: {{?user_query}}\n
                Context: {{?grounding_response}}""")
    ]
)


### Define the LLM

In [30]:
from gen_ai_hub.orchestration_v2.models.llm_model_details import LLMModelDetails

llm = LLMModelDetails(name="gpt-4o", params={"max_completion_tokens": 2048})


In [ ]:
from gen_ai_hub.orchestration_v2.models.document_grounding import (GroundingModuleConfig,GroundingType,
DocumentGroundingFilter,DataRepositoryType,DocumentGroundingConfig,DocumentGroundingPlaceholders,GroundingSearchConfig)

filters=[DocumentGroundingFilter(id="vector",
                                   data_repositories=["a0165*************10855f"],
                                   data_repository_type=DataRepositoryType.VECTOR.value,
                                   search_config= GroundingSearchConfig(max_chunk_count=20)
                                   )]


placeholders = DocumentGroundingPlaceholders(
    input=["user_query"],
    output="grounding_response"
)

# Grounding module config
grounding_config = GroundingModuleConfig(
    type=GroundingType.DOCUMENT_GROUNDING_SERVICE.value,
    config=DocumentGroundingConfig(
        filters=filters,
        placeholders=placeholders
    )
)

In [32]:
from gen_ai_hub.orchestration_v2.models.template import PromptTemplatingModuleConfig
from gen_ai_hub.orchestration_v2.models.config import ModuleConfig, OrchestrationConfig

prompt_template = PromptTemplatingModuleConfig(prompt=template,
                                               model=llm)

module_config = ModuleConfig(prompt_templating=prompt_template, grounding = grounding_config)

config = OrchestrationConfig(modules=module_config)


 #### Step 3: Generate context-relevant answer for a user query
   - We now invoke the orchestration service by providing a user query. The query is grounded against the document index, and the LLM uses the grounding result to generate an informed response.

In [33]:
from gen_ai_hub.proxy import get_proxy_client
from gen_ai_hub.orchestration_v2.service import OrchestrationService

proxy_client = get_proxy_client()

orchestration_service = OrchestrationService(
    proxy_client=proxy_client,
    config=config
)


In [34]:
response = orchestration_service.run(placeholder_values={"user_query": "Is there any complaint?"})
print(response.final_result.choices[0].message.content)


Yes, there are complaints in the context provided. Here are the complaints identified:

1. **Window Cleaning Service Oversight**: Mark Phillips reported that the windows in the east wing of the Riverfront Business Complex were missed during the last cleaning service, and this has been a recurring issue.

2. **Landscaping Service Issues**: James Anderson reported that the recent landscaping service at Crestview Gardens Apartments was not performed properly, with shrubs not being trimmed correctly and debris left behind.

These complaints should be addressed promptly to ensure customer satisfaction.


In [35]:
print(response)

request_id='bf3d5b3d-c894-9994-88fb-b7c94991230a' intermediate_results=ModuleResults(grounding=GenericModuleResult(message='grounding result', data={'grounding_query': 'grounding call', 'grounding_result': "Subject: Feedback on HVAC Repair\n\nHello,\n\nI wanted to thank your team for the prompt repair of our HVAC system at Lakeview Corporate Offices. However, there’s been a minor noise issue since. Is it possible to have a technician look into this?\n\nWarm regards,\nRobert Kim```Subject: Complaint: Window Cleaning Service Oversight\n\nDear Facility Solutions,\n\nI wanted to bring to your attention that the windows in the east wing of the Riverfront Business Complex were missed during the last cleaning service. This has been recurring, and I would appreciate it if we could resolve this soon.\n\nRegards,\nMark Phillips```Subject: Complaint About Recent Landscaping Service\n\nDear Facility Solutions Team,\n\nI’m writing to report some issues with the recent landscaping service at Crestvi